# Critical Stations in Israel Public Transportation

Project title: תחנות קריטיות: ניתוח מרכזיות ועמידות ברשת התחבורה הציבורית בישראל

This notebook runs the project pipeline that builds a GTFS stop graph, computes centrality metrics, simulates resilience under stop removals, and loads report-ready outputs.

## Graph Model

- Node: a GTFS stop from `stops.txt` that appears in `stop_times.txt`.
- Directed edge: two consecutive stops in the same trip.
- Edge weight: number of scheduled trip segments using that ordered stop pair.
- Undirected graph: used for connectivity, articulation points, bridges, and resilience.
- Directed graph: used for in/out degree and weighted PageRank.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "israel-public-transportation"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Sanity check: the large GTFS table must be present before the full run.
# It is not tracked in git - see the README for the one-line download.
path = DATA_DIR / "stop_times.txt"
if not path.exists():
    raise FileNotFoundError(
        f"{path} is missing. Download it first (see README):\n"
        f'  pip install gdown\n'
        f'  gdown <GOOGLE_DRIVE_FILE_ID> -O "{path.as_posix()}"'
    )
prefix = path.read_bytes()[:80]
assert not prefix.startswith(b"version https://git-lfs.github.com/spec/v1"), (
    f"{path} is a leftover Git LFS pointer from an older clone. "
    f"This repository no longer uses Git LFS - delete it and download the real file."
)
print("stop_times.txt", round(path.stat().st_size / 1024**2, 1), "MB")

The full run streams `stop_times.txt`, computes centrality, shortest-path, resilience, accessibility, community, and network-model outputs. For a faster exploratory run, lower the sample counts or set optional samples to `0`.

In [ ]:
BETWEENNESS_SAMPLES = 128
HARMONIC_SAMPLES = 512
PATH_SOURCE_SAMPLES = 128
CLUSTERING_TRIALS = 2000
RESILIENCE_REMOVALS = 500
RESILIENCE_STEPS = 25
ACCESSIBILITY_PAIRS = 300
ACCESSIBILITY_REMOVALS = 500
ACCESSIBILITY_STEPS = 10
RANDOM_TRIALS = 5

cmd = [
    sys.executable,
    "src/transit_network_analysis.py",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--betweenness-samples", str(BETWEENNESS_SAMPLES),
    "--harmonic-samples", str(HARMONIC_SAMPLES),
    "--path-source-samples", str(PATH_SOURCE_SAMPLES),
    "--clustering-trials", str(CLUSTERING_TRIALS),
    "--resilience-removals", str(RESILIENCE_REMOVALS),
    "--resilience-steps", str(RESILIENCE_STEPS),
    "--accessibility-pairs", str(ACCESSIBILITY_PAIRS),
    "--accessibility-removals", str(ACCESSIBILITY_REMOVALS),
    "--accessibility-steps", str(ACCESSIBILITY_STEPS),
    "--random-trials", str(RANDOM_TRIALS),
]
subprocess.run(cmd, check=True)

## Network Summary

In [ ]:
summary = pd.read_csv(OUTPUT_DIR / "tables/network_summary.csv")
build_summary = pd.read_csv(OUTPUT_DIR / "tables/gtfs_graph_build_summary.csv")

display(build_summary.T.rename(columns={0: "value"}))
display(summary.T.rename(columns={0: "value"}))

## Critical Stop Rankings

In [ ]:
ranking_files = {
    "Degree": "top_degree_stops.csv",
    "Weighted Degree": "top_weighted_degree_stops.csv",
    "PageRank": "top_pagerank_stops.csv",
    "Approx Betweenness": "top_approx_betweenness_stops.csv",
    "Approx Harmonic": "top_approx_harmonic_stops.csv",
    "Articulation Points": "top_articulation_points.csv",
}

for title, filename in ranking_files.items():
    print("\n", title)
    cols = [
        "stop_id", "stop_name", "degree", "weighted_degree",
        "pagerank", "approx_betweenness", "approx_harmonic",
        "is_articulation_point",
    ]
    frame = pd.read_csv(OUTPUT_DIR / "tables" / filename)
    display(frame[[column for column in cols if column in frame.columns]].head(10))

## Resilience Results

In [ ]:
resilience = pd.read_csv(OUTPUT_DIR / "tables/resilience_random_vs_targeted.csv")
display(resilience.head())
display(resilience.groupby("strategy").tail(1).sort_values("largest_component_share"))

accessibility_path = OUTPUT_DIR / "tables/accessibility_damage_by_removal.csv"
if accessibility_path.exists():
    accessibility = pd.read_csv(accessibility_path)
    display(accessibility.head())
    display(accessibility.groupby("strategy").tail(1).sort_values("reachable_share"))

## Communities

In [ ]:
communities = pd.read_csv(OUTPUT_DIR / "tables/community_summary.csv")
display(communities.head(15))

## Report Figures

In [ ]:
for figure in [
    "active_stops_map.png",
    "top_weighted_degree_stops.png",
    "top_pagerank_stops.png",
    "route_type_distribution.png",
    "top_communities.png",
    "resilience_curve.png",
]:
    print(figure)
    display(Image(filename=str(OUTPUT_DIR / "figures" / figure)))

## Interpretation Checklist For The Final Report

- Compare degree, weighted degree, PageRank, and approximate betweenness rankings.
- Explain whether high-frequency transfer corridors differ from structurally critical articulation points.
- Use the resilience curve to compare targeted removals against random removals.
- Discuss limitations: scheduled service is not passenger demand, GTFS stop granularity may split nearby platforms, and exact betweenness is approximated for scalability.

## Network Model Comparison

In [ ]:
model_path = OUTPUT_DIR / "tables/network_model_comparison.csv"
if model_path.exists():
    model_comparison = pd.read_csv(model_path)
    display(model_comparison)

degree_dist_path = OUTPUT_DIR / "tables/degree_distribution.csv"
if degree_dist_path.exists():
    degree_distribution = pd.read_csv(degree_dist_path)
    display(degree_distribution.head(10))